# 4章 データからコンテキストへ - LCMのケーススタディ：引用付きクイズ生成

## 事前準備

当サンプルコードを実行するには、GeminiのAPIキーが必要です。

### Gemini

- Google AI StudiohのWebページ（ https://aistudio.google.com/api-keys ）にアクセスし、ご自身のアカウントでログインして、APIキーを生成してください。
- 生成したAPIキーを定数 `GEMINI_API_KEY` に設定します。

```
GEMINI_API_KEY=<作成したAPIキー>
```


In [ ]:
GEMINI_API_KEY= "<作成したAPIキー>"

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [ ]:
!pip install markitdown google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.9 MB/s eta 0:00:00


In [ ]:
from markitdown import MarkItDown
from typing import List, Dict, Optional, Any

import os

from google import genai
from google.genai import types
from langsmith import Client as LangSmithClient

PROMPT_TEMPLATE = "souzatharsis/duo_msg"
PROMPT_COMMIT = "2f61756a"

PROMPT_QUIZ_TEMPLATE = "souzatharsis/duo_quiz"
PROMPT_QUIZ_COMMIT = "9e9a7500"

GEMINI_MODEL_NAME = "gemini-3.1-pro-preview"
GEMINI_API_KEY_LABEL = "GEMINI_API_KEY"

MIN_CACHE_TOKENS = 1024


def render_langsmith_prompt(prompt_template: Any, variables: Dict[str, Any]) -> str:
    """
    LangSmithのプロンプトテンプレートの結果をプレーンテキストに変換する。
    """
    prompt = prompt_template.invoke(variables)

    if hasattr(prompt, "messages"):
        return "\n".join(
            getattr(message, "content", str(message))
            for message in prompt.messages
        )

    if hasattr(prompt, "text"):
        return prompt.text

    if hasattr(prompt, "to_string"):
        return prompt.to_string()

    return str(prompt)


class LLMBackend:
    """
    キャッシュ済みコンテンツを使用できるGemini生成用のバックエンドクラス。
    """

    CACHE_TTL = 60  # キャッシュの有効期間（分）

    def __init__(
        self,
        model_name: str = GEMINI_MODEL_NAME,
        api_key_label: str = GEMINI_API_KEY_LABEL,
        conversation_config: Optional[Dict] = None,
        input: str = "",
        cache_ttl: int = CACHE_TTL,
    ):
        """
        LLMBackendを初期化する。

        Args:
            model_name (str): 使用するモデルの名前
            api_key_label (str): Gemini APIキーを格納した環境変数名
            conversation_config (Optional[Dict]): 会話設定
            input (str): メモリまたはキャッシュ元として使用する入力コンテンツ
            cache_ttl (int): キャッシュの有効期間。単位は分
        """
        self.model_name = model_name
        self.conversation_config = conversation_config or {}
        self.client = genai.Client(api_key=os.environ[api_key_label])

        self.system_instruction = self.compose_prompt(
            input=input,
            conversation_config=self.conversation_config,
        )

        self.cache = None
        self.cache_source = input.strip()

        if self._can_create_cache(self.cache_source):
            self.cache = self.client.caches.create(
                model=self.model_name,
                config=types.CreateCachedContentConfig(
                    display_name="duo_knowledge_base",
                    system_instruction=self.system_instruction,
                    contents=self.cache_source,
                    ttl=f"{cache_ttl * 60}s",
                ),
            )

    def _can_create_cache(self, text: str) -> bool:
        if not text:
            return False

        token_count = self.client.models.count_tokens(
            model=self.model_name,
            contents=text,
        ).total_tokens

        return token_count >= MIN_CACHE_TOKENS

    def compose_prompt(self, input: str, conversation_config: Dict) -> str:
        """
        LangSmithのプロンプトテンプレートを使用してシステムプロンプトを構成する。
        """
        langsmith_client = LangSmithClient()

        prompt_template = langsmith_client.pull_prompt(
            f"{PROMPT_TEMPLATE}:{PROMPT_COMMIT}",
            dangerously_pull_public_prompt=True,
        )

        return render_langsmith_prompt(
            prompt_template,
            {
                "memory": input,
                "input_texts": "",
            },
        )

    def generate_content(self, contents: str):
        if self.cache is not None:
            config = types.GenerateContentConfig(
                cached_content=self.cache.name,
            )
        else:
            config = types.GenerateContentConfig(
                system_instruction=self.system_instruction,
            )

        return self.client.models.generate_content(
            model=self.model_name,
            contents=contents,
            config=config,
        )


class Quiz:
    """
    キャッシュ済みコンテンツを使用できるクイズ生成用のバックエンドクラス。
    """

    CACHE_TTL = 60  # キャッシュの有効期間（分）

    def __init__(
        self,
        model_name: str = GEMINI_MODEL_NAME,
        api_key_label: str = GEMINI_API_KEY_LABEL,
        input: str = "",
        cache_ttl: int = CACHE_TTL,
        add_citations: bool = False,
        num_questions: int = 10,
    ):
        """
        Quizを初期化する。

        Args:
            model_name (str): 使用するモデルの名前
            api_key_label (str): Gemini APIキーを格納した環境変数名
            input (str): クイズ生成元となる入力コンテンツ
            cache_ttl (int): キャッシュの有効期間。単位は分
            add_citations (bool): クイズの回答に引用を含めるかどうか
            num_questions (int): 生成するクイズの問題数
        """
        self.model_name = model_name
        self.input = input
        self.client = genai.Client(api_key=os.environ[api_key_label])

        self.system_instruction = self.compose_prompt(
            num_questions=num_questions,
            add_citations=add_citations,
        )

        self.cache = None
        self.cache_source = input.strip()

        if self._can_create_cache(self.cache_source):
            self.cache = self.client.caches.create(
                model=self.model_name,
                config=types.CreateCachedContentConfig(
                    display_name="duo_quiz",
                    system_instruction=self.system_instruction,
                    contents=self.cache_source,
                    ttl=f"{cache_ttl * 60}s",
                ),
            )

    def _can_create_cache(self, text: str) -> bool:
        if not text:
            return False

        token_count = self.client.models.count_tokens(
            model=self.model_name,
            contents=text,
        ).total_tokens

        return token_count >= MIN_CACHE_TOKENS

    def compose_prompt(
        self,
        num_questions: int = 10,
        add_citations: bool = True,
    ) -> str:
        if add_citations:
            citations = (
                "In the Quiz answers, at the end, make sure to add Input ID "
                "indicating referenced content pertaining to the answer to the question."
            )
        else:
            citations = ""

        langsmith_client = LangSmithClient()

        prompt_template = langsmith_client.pull_prompt(
            f"{PROMPT_QUIZ_TEMPLATE}:{PROMPT_QUIZ_COMMIT}",
            dangerously_pull_public_prompt=True,
        )

        return render_langsmith_prompt(
            prompt_template,
            {
                "num_questions": num_questions,
                "citations": citations,
                "input_texts": "",
            },
        )

    def generate(self, msg: str = "") -> str:
        msg = f"Generate: {msg}"

        if self.cache is not None:
            contents = msg
            config = types.GenerateContentConfig(
                cached_content=self.cache.name,
            )
        else:
            contents = f"""
INPUT:
{self.input}

USER_INSTRUCTIONS:
{msg}
"""
            config = types.GenerateContentConfig(
                system_instruction=self.system_instruction,
            )

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=contents,
            config=config,
        )
        print("cached tokens:", response.usage_metadata.cached_content_token_count)

        return response.text


class ContentGenerator:
    """
    LLMBackendを使用してコンテンツを生成するクラス。
    """

    def __init__(self, llm: LLMBackend):
        """
        モデルと入力コンテンツを指定してContentGeneratorを初期化する。

        Args:
            llm: 生成に使用するLLMBackendインスタンス。
        """
        self.llm = llm

    # キャッシュを使用した生成
    def generate(self, input_content: str, user_instructions: str = ""):
        """
        モデルを使用してコンテンツを生成する。

        Returns:
            モデルからの応答。
        """

        if self.llm.cache is not None:
            prompt = f"""
USER_INSTRUCTIONS:
Make sure to follow these instructions: {user_instructions}

Use the cached input content as the main reference.
"""
        else:
            prompt = f"""
USER_INSTRUCTIONS:
Make sure to follow these instructions: {user_instructions}

INPUT:
{input_content}
"""

        return self.llm.generate_content(prompt)


class Client:
    def __init__(self, knowledge_base: Optional[List[str]] = None):
        """
        会話設定を使用してClientクラスを初期化する。
        """
        self.knowledge_base = knowledge_base or []  # ユーザーが指定する基礎知識。URLのリストで表す
        self.reference_id = 0  # 各入力に付与する一意のID

        self.input = ""  # 短期メモリ。現在学習対象としている入力
        self.urls = []  # コンテンツ抽出元となるURLの入力リスト
        self.response = ""  # LLMからの最新の応答

        self.urls_memory = []  # コンテンツ抽出元URLの累積リスト
        self.input_memory = ""  # 長期メモリ。入力とナレッジベースの累積
        self.response_memory = ""  # 長期的な応答メモリ。応答の累積

        self.extractor = MarkItDown()  # URLからコンテンツを抽出するための抽出器
        self.quiz_instance = None

        self.add_knowledge_base(self.knowledge_base)

        self.llm = LLMBackend(input=self.input_memory)
        self.content_generator = ContentGenerator(llm=self.llm)

    def add_knowledge_base(self, urls: List[str]) -> None:
        """
        ナレッジベースにURLを追加する。
        """
        self.add(urls)

    def add(self, urls: List[str]) -> None:
        """
        URLからコンテンツを抽出し、会話入力に追加する。

        Args:
            urls (List[str]): コンテンツ抽出元となるURLのリスト。
        """
        self.urls = urls

        # 引用を可能にするために、CiC形式に従って新しいコンテンツを入力へ追加する
        for url in urls:
            self.urls_memory.append(url)

            content = self.extractor.convert(url).text_content
            formatted_content = (
                f"ID: {self.reference_id} | {content} | END ID: {self.reference_id}"
            )

            self.input += formatted_content + "\n"
            self.reference_id += 1

        # メモリを更新する
        self.input_memory = self.input_memory + self.input

    def msg(self, msg: str = "", add_citations: bool = False) -> str:
        """
        現在の会話入力に基づいてQ&Aコンテンツを生成する。

        Args:
            msg (str): コンテンツ生成のための追加指示。
            add_citations (bool): 応答に入力の引用を含めるかどうか。
                デフォルトはFalse。

        Returns:
            str: 生成されたQ&A会話コンテンツ。
        """
        if add_citations:
            msg = msg + "\n\nFor key statements, add Input ID to the response."

        response = self.content_generator.generate(
            input_content=self.input,
            user_instructions=msg,
        )
        self.response = response.text
        self.response_memory = self.response_memory + self.response

        return self.response

    def quiz(
        self,
        add_citations: bool = True,
        num_questions: int = 10,
    ) -> Quiz:
        """
        入力メモリ全体に基づいてクイズを生成する。
        """
        self.quiz_instance = Quiz(
            input=self.input_memory,
            add_citations=add_citations,
            num_questions=num_questions,
        )

        return self.quiz_instance


In [ ]:
kb = [f"https://www.gutenberg.org/cache/epub/{i}/pg{i}.txt" for i in range(1, 10)]

duo = Client(knowledge_base=kb)

In [ ]:
study_references = [
    "https://www.gutenberg.org/cache/epub/10000/pg10000.txt",
    "https://www.gutenberg.org/cache/epub/65363/pg65363.txt"
]
duo.add(study_references)
quiz = duo.quiz(add_citations=True)

In [ ]:
# 訳注： 日本語訳版に向けて、なるべく日本語で生成されるように指示している。ただし、完全ではない。
generated_quiz = quiz.generate("日本語で出題してください。")

cached tokens: 470658


In [ ]:
from rich.markdown import Markdown
display(Markdown(generated_quiz))

ハーバード大学の歴史・政治学の講義へようこそ。本日は、提供された歴史的文献（アメリカの建国史およびイギリス憲政史を 
形作る重要な文書群）に基づき、皆さんの理解度と学術的な分析力を試す上級者向けのクイズを10問用意しました。           

これらの問題は、単なる事実の暗記ではなく、歴史的背景や法的な解釈、文書間の関連性を深く理解しているかを問うものです 
。各問題にじっくりと取り組み、最適な選択肢を選んでください。解答と参照元のInput                                    
IDは、クイズの最後にまとめて記載してあります。                                                                     

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                         高度歴史・憲政史クイズ（全10問）                                          

問題 1 アメリカ独立宣言において、イギリス国王が代表議会（Representative                                            
Houses）を繰り返し解散させた結果、破壊不能（incapable of                                                           
Annihilation）な性質を持つ「立法権」はどのような状態に置かれたと非難されているか？ A.                              
植民地総督によって一時的に代行される状態に置かれた。 B. 国王の直接の管理下に置かれ、専制的な行使を受けた。 C.      
イギリス本国の議会に吸収され、植民地の自治が失われた。 D. 人民全体に返還され、彼らによって行使される状態となった。 

問題 2 アメリカ合衆国権利章典（修正第5条）によれば、大陪審（Grand                                                  
Jury）の告発または起訴なしに死刑または破廉恥罪に問われることがないとする原則には例外が設けられている。その例外に該 
当するのはどのようなケースか？ A. 連邦議会の両院が3分の2の賛成で国家反逆罪と認定した場合。 B.                      
陸海軍、または戦時や公共の危険に際して現に服務中の民兵において発生した事件。 C.                                    
大統領によって国家非常事態（National Emergency）が宣言された期間中の事件。 D.                                      
外国政府の諜報員によるスパイ行為が明白な場合。                                                                     

問題 3                                                                                                             
ジョン・F・ケネディの大統領就任演説において、彼はある人々に対して「共産主義者が行っているからでも、彼らの票を求めて
いるからでもなく、それが正しいからだ」という理由で、彼らが自らを助けるための最善の努力を支援すると誓っている。この 
対象となっているのは誰か？ A. ヨーロッパの古き同盟国で復興を目指す人々 B.                                          
自由の列に新たに加わった新興国家の指導者たち C.                                                                    
世界の半分の小屋や村で、大量の悲惨な絆を断ち切ろうと奮闘している人々 D.                                            
南の国境を接し、「進歩のための同盟」を結ぶ姉妹共和国の人々                                                         

問題 4                                                                                                             
アメリカ合衆国憲法において、大統領が弾劾裁判（Impeachment）を受ける際、上院での裁判を主宰する役職、および有罪判決を
下すために必要な条件の組み合わせとして正しいものはどれか？ A. 副大統領が主宰し、出席議員の過半数の同意が必要。 B.  
上院仮議長（President pro tempore）が主宰し、全上院議員の3分の2の同意が必要。 C. 最高裁判所長官（Chief             
Justice）が主宰し、出席議員の3分の2の同意が必要。 D.                                                               
最高裁判所長官が主宰し、上院および下院の合同会議で過半数の同意が必要。                                             

問題 5 パトリック・ヘンリーの有名な「自由か死か」の演説において、彼が自らの歩みを導く「唯一のランプ（one           
lamp）」と呼び、将来を判断するための基準としたものは何か？ A. 神の摂理（Providence） B. 理性と論理（Reason and     
Logic） C. 経験（Experience） D. 人間の自然権（Natural Rights of Man）                                             

問題 6 1620年に結ばれた「メイフラワー誓約（The Mayflower                                                           
Compact）」において、彼らが航海を行い、最初の植民地を建設しようとしていた目的地（地域）はどこであると記載されている
か？ A. マサチューセッツ湾（Massachusetts Bay） B. ヴァージニアの北部（the Northerne Parts of Virginia） C.        
プリマス・ロック（Plymouth Rock） D. ニューイングランド沿岸（the coast of New England）                            

問題 7                                                                                                             
エイブラハム・リンカーンの「ゲティスバーグ演説」の中で、彼は「我々がこの地を奉献（dedicate）したり、神聖なもの（con
secrate）にしたり、清めたり（hallow）することはできない」と述べている。その理由として彼が挙げているのはどれか？ A. 
神の意志のみが、戦争で失われた命の代償として土地を清めることができるから。 B.                                      
この戦争が終結し、自由の新たな誕生を迎えるまで、真の奉献は完了しないから。 C.                                      
ここで戦った勇敢な人々（生者も死者も）が、我々の貧弱な力をはるかに超えて、すでにこの地を神聖なものにしているから。 
D. 同胞の血で汚れた土地は、いかなる人間の言葉や儀式によっても浄化することができないから。                          

問題 8                                                                                                             
エイブラハム・リンカーンの「第2次大統領就任演説」において、南北戦争のそもそもの原因となった「特異で強力な利益（pecu
liar and powerful                                                                                                  
interest）」を構成していたのは何か、またそれは当時の全人口のどの程度を占めていたとリン